# Day 36: Confidence Intervals

## What Is a Confidence Interval?
A single sample mean is just one number, and it will almost never exactly equal the true population mean - a confidence interval (CI) gives you a range around that estimate that quantifies how much uncertainty there is, rather than pretending the point estimate is exact. A "95% confidence interval" means that if you repeated the sampling process many times and built a CI each time, about 95% of those intervals would contain the true population parameter - it is a statement about the reliability of the method, not a 95% probability that this specific interval contains the true value. This matters in real data science whenever you report a metric estimated from a sample (average order value, click-through rate, etc.) and need to communicate how much to trust that number. The code below draws a sample of 40 values from a normal distribution (mean 100, standard deviation 15) that we'll use to build a CI in the next cell, and prints its sample mean as our starting point estimate.

In [ ]:
import numpy as np
from scipy import stats

# Fix the seed for reproducible random sampling
np.random.seed(1)
# Simulate a sample of 40 values from a normal distribution (true mean=100, std=15)
sample = np.random.normal(loc=100, scale=15, size=40)
print('Sample mean:', sample.mean())

## Calculating a 95% CI for the Mean
Building a CI for a mean requires three ingredients: the sample mean (our best point estimate), the standard error (how much sample means would be expected to vary from sample to sample), and a critical value from an appropriate distribution that sets how wide the interval needs to be for the desired confidence level. `stats.sem` computes the standard error of the mean directly from the sample. We use the t-distribution (via `stats.t.interval`) rather than the normal distribution because the sample size here is small and the true population standard deviation is unknown - the t-distribution has heavier tails to account for that extra uncertainty, and it needs the degrees of freedom (`len(sample) - 1`) as a parameter. The result is a `(lower, upper)` tuple - the range we're 95% confident contains the true population mean, given what this sample showed us.

In [ ]:
# Point estimate and its standard error (spread expected across repeated samples)
mean = sample.mean()
sem = stats.sem(sample)
# t.interval builds the CI using the t-distribution (better than normal for small n);
# degrees of freedom = sample size - 1
ci = stats.t.interval(0.95, len(sample) - 1, loc=mean, scale=sem)
print('95% Confidence Interval:', ci)

## Effect of Sample Size
The standard error shrinks as sample size grows (it's proportional to 1/sqrt(n)), which directly narrows the resulting confidence interval - in other words, more data gives you a more precise estimate of the population mean, even though the confidence level (95%) stays the same. This is a key intuition for data science practice: if a CI is too wide to be useful for a business decision, the fix is usually to collect more data, not to lower your confidence level. The loop below draws three fresh samples of increasing size (10, 40, and 200), computes a 95% CI for each, and prints the width of that interval (`ci[1] - ci[0]`). Reading the output top to bottom, the CI width should shrink noticeably as `n` increases, showing the interval tightening around the true mean as more data is collected.

In [ ]:
# Compare CI width across increasing sample sizes - larger n should narrow the interval
for n in [10, 40, 200]:
    s = np.random.normal(loc=100, scale=15, size=n)
    ci = stats.t.interval(0.95, n - 1, loc=s.mean(), scale=stats.sem(s))
    print(f'n={n}: CI width = {ci[1] - ci[0]:.2f}')

## Confidence Interval for a Proportion
Not all data is a continuous measurement like height or spend - a lot of real business data is binary (yes/no, converted/didn't, defective/not), and the metric of interest is a proportion or rate, like "what percentage of customers prefer this product?" The math differs slightly from the mean case: the standard error of a proportion is calculated as `sqrt(p_hat * (1 - p_hat) / n)`, where `p_hat` is the observed sample proportion, and for large samples we can use the fixed z-value 1.96 (instead of a t-distribution critical value) because the sampling distribution of a proportion is approximately normal for reasonably large n. Below, out of 100 people surveyed, 68 said yes, giving `p_hat = 0.68`; the CI is then built by adding and subtracting `1.96 * se` from that proportion, giving a range we're 95% confident contains the true population proportion.

In [ ]:
# 68 out of 100 surveyed said "yes" - p_hat is the observed sample proportion
successes, n = 68, 100
p_hat = successes / n
# Standard error of a proportion
se = (p_hat * (1 - p_hat) / n) ** 0.5
# 1.96 is the z-critical value for a 95% confidence level (large-sample approximation)
ci = (p_hat - 1.96 * se, p_hat + 1.96 * se)
print('95% CI for proportion:', ci)